<a href="https://colab.research.google.com/github/NadiaJeni/ArrayCategoriesTest/blob/main/baseline_fedann_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ===== Cell 1: Setup & Global Config =====

# Core ML
!pip install --quiet scikit-learn lightgbm openpyxl opacus

import os, json, numpy as np, pandas as pd
import torch
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import roc_auc_score, average_precision_score

# ---------- Global Paths ----------
BASE_DIR = "/content"
PSEUDO_DIR = f"{BASE_DIR}/pseudo_label_output"
FED_SIM_DIR = f"{BASE_DIR}/fed_sim"
FED_OUT_DIR = f"{BASE_DIR}/fed_output"
EMB_DIR = f"{BASE_DIR}/embeddings"
ENS_DIR = f"{BASE_DIR}/ensemble_output"

for d in [PSEUDO_DIR, FED_SIM_DIR, FED_OUT_DIR, EMB_DIR, ENS_DIR]:
    os.makedirs(d, exist_ok=True)

# ---------- Experiment Toggles ----------
NUM_CLIENTS = 20
TOP_PCT =0.001          # 🔥 tuned: stronger pseudo-label precision
ISO_CONTAMINATION = 0.001

# ---------- Federated ANN ----------
N_ROUNDS = 12
LOCAL_EPOCHS = 1
LR = 1e-3
BATCH_SIZE = 512
EMB_SIZE = 64

# ---------- Differential Privacy ----------
USE_DP = False            # 🔴 change to True later
NOISE_MULTIPLIER = 1.0
MAX_GRAD_NORM = 1.0
DELTA = 1e-5

print("✅ Setup complete")
print("DP Enabled:", USE_DP)


✅ Setup complete
DP Enabled: False


In [2]:
# ===== Cell 2: Data Load + Feature Engineering =====

from google.colab import files

print("📂 Upload fraudTest.csv or fraudTest.xlsx")
uploaded = files.upload()
fname = list(uploaded.keys())[0]

if fname.endswith(".xlsx"):
    df = pd.read_excel(fname)
else:
    df = pd.read_csv(fname)

print("Raw shape:", df.shape)

# ---------- Feature Engineering ----------
def feature_engineering(df):
    df = df.copy()

    df['trans_date_trans_time'] = pd.to_datetime(
        df.get('trans_date_trans_time'), errors='coerce'
    )
    df['dob'] = pd.to_datetime(df.get('dob'), errors='coerce')

    df['hour'] = df['trans_date_trans_time'].dt.hour.fillna(0)
    df['age'] = ((df['trans_date_trans_time'] - df['dob'])
                 .dt.days / 365.25).fillna(40)

    def haversine(r):
        try:
            from math import radians, sin, cos, sqrt, atan2
            R = 6371
            lat1, lon1, lat2, lon2 = map(
                radians, [r.lat, r.long, r.merch_lat, r.merch_long]
            )
            dlat, dlon = lat2-lat1, lon2-lon1
            a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
            return 2 * R * atan2(sqrt(a), sqrt(1-a))
        except:
            return 0

    df['dist_km'] = df.apply(haversine, axis=1)
    df['high_amt'] = (df['amt'] >= 1000).astype(int)
    df['night_txn'] = ((df['hour'] >= 22) | (df['hour'] < 6)).astype(int)
    df['far_merchant'] = (df['dist_km'] > 50).astype(int)
    df['old_and_big_amt'] = ((df['age'] >= 65) & (df['amt'] >= 500)).astype(int)

    return df

df = feature_engineering(df)
print("After FE shape:", df.shape)

assert 'is_fraud' in df.columns, "❌ is_fraud column missing"
print("True frauds:", df['is_fraud'].sum())


📂 Upload fraudTest.csv or fraudTest.xlsx


Saving fraudTest.csv.xlsx to fraudTest.csv.xlsx
Raw shape: (555719, 23)
After FE shape: (555719, 30)
True frauds: 2145


In [5]:
# ===== Cell 3: Pseudo-Label Generation =====

from sklearn.ensemble import IsolationForest

# ---------- Weak Rules ----------
rule = (
    (df['amt'] > 5000).astype(int) * 2 +
    (df['amt'] > 2000).astype(int) +
    df['night_txn'] +
    df['far_merchant'] +
    ((df['age'] < 18) | (df['age'] > 80)).astype(int)
)

# ---------- Isolation Forest ----------
feat_cols = [
    'amt','city_pop','age','dist_km','hour',
    'high_amt','night_txn','far_merchant','old_and_big_amt'
]

X_iso = StandardScaler().fit_transform(df[feat_cols].fillna(0))
iso = IsolationForest(
    n_estimators=200,
    contamination=ISO_CONTAMINATION,
    random_state=42
)
anom = -iso.fit(X_iso).decision_function(X_iso)

# ---------- Meta Score ----------
sc = MinMaxScaler()
meta_score = sc.fit_transform(
    np.c_[rule.values, anom]
).mean(axis=1)

df['meta_score'] = meta_score

# ---------- Select Top PCT ----------
top_k = int(TOP_PCT * len(df))
idx = np.argsort(meta_score)[-top_k:]

df['pseudo_label'] = 0
df.loc[df.index[idx], 'pseudo_label'] = 1

# ---------- Evaluation vs True ----------
tp = ((df.pseudo_label==1)&(df.is_fraud==1)).sum()
fp = ((df.pseudo_label==1)&(df.is_fraud==0)).sum()
fn = ((df.pseudo_label==0)&(df.is_fraud==1)).sum()

precision = tp / (tp+fp)
recall = tp / (tp+fn)

print("Pseudo positives:", df.pseudo_label.sum())
print("TP:", tp, "FP:", fp, "FN:", fn)
print("Precision:", round(precision,3))
print("Recall:", round(recall,3))

df.to_csv(f"{PSEUDO_DIR}/full_with_pseudo_labels.csv", index=False)
print("✅ Saved pseudo-labeled dataset")


Pseudo positives: 555
TP: 121 FP: 434 FN: 2024
Precision: 0.218
Recall: 0.056
✅ Saved pseudo-labeled dataset


In [6]:
# ===== Cell 4: Federated Client Split =====

from sklearn.preprocessing import StandardScaler
import json

df = pd.read_csv(f"{PSEUDO_DIR}/full_with_pseudo_labels.csv")

FEATURE_COLS = [
    'amt','city_pop','age','dist_km','hour',
    'high_amt','night_txn','far_merchant','old_and_big_amt'
]

X = df[FEATURE_COLS].fillna(0)
y = df['pseudo_label'].values

# ---------- Scale features ----------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save preprocessing metadata
with open(f"{FED_SIM_DIR}/preproc_meta.json", "w") as f:
    json.dump({"feature_cols": FEATURE_COLS}, f, indent=2)

import joblib
joblib.dump(scaler, f"{FED_SIM_DIR}/scaler.pkl")

# ---------- Shuffle & split ----------
idx = np.random.permutation(len(df))
splits = np.array_split(idx, NUM_CLIENTS)

clients_info = {}

for i, ids in enumerate(splits):
    client_df = df.iloc[ids].copy()
    path = f"{FED_SIM_DIR}/client_{i:02d}.csv"
    client_df.to_csv(path, index=False)

    clients_info[i] = {
        "rows": len(client_df),
        "pseudo_pos": int(client_df['pseudo_label'].sum()),
        "path": path
    }

    print(f"Client {i:02d}: rows={len(client_df)}, pseudo_pos={clients_info[i]['pseudo_pos']}")

# ---------- Save client metadata ----------
with open(f"{FED_SIM_DIR}/clients_info.json", "w") as f:
    json.dump(clients_info, f, indent=2)

print("\n✅ Clients ready for Federated Learning")
print("Total pseudo positives:", df['pseudo_label'].sum())


Client 00: rows=27786, pseudo_pos=32
Client 01: rows=27786, pseudo_pos=37
Client 02: rows=27786, pseudo_pos=34
Client 03: rows=27786, pseudo_pos=20
Client 04: rows=27786, pseudo_pos=32
Client 05: rows=27786, pseudo_pos=30
Client 06: rows=27786, pseudo_pos=31
Client 07: rows=27786, pseudo_pos=31
Client 08: rows=27786, pseudo_pos=28
Client 09: rows=27786, pseudo_pos=23
Client 10: rows=27786, pseudo_pos=25
Client 11: rows=27786, pseudo_pos=26
Client 12: rows=27786, pseudo_pos=33
Client 13: rows=27786, pseudo_pos=21
Client 14: rows=27786, pseudo_pos=19
Client 15: rows=27786, pseudo_pos=20
Client 16: rows=27786, pseudo_pos=29
Client 17: rows=27786, pseudo_pos=27
Client 18: rows=27786, pseudo_pos=32
Client 19: rows=27785, pseudo_pos=25

✅ Clients ready for Federated Learning
Total pseudo positives: 555


In [7]:
# ===== Cell 5: Federated ANN (FedAvg, optional DP-SGD) =====

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from opacus import PrivacyEngine
import json, time, copy

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ---------- Simple ANN ----------
class FedANN(nn.Module):
    def __init__(self, in_dim, emb_dim=64):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 128)
        self.fc2 = nn.Linear(128, emb_dim)
        self.out = nn.Linear(emb_dim, 1)

    def forward(self, x, return_emb=False):
        x = F.relu(self.fc1(x))
        emb = F.relu(self.fc2(x))
        logit = self.out(emb)
        if return_emb:
            return logit, emb
        return logit


# ---------- Load client metadata ----------
with open(f"{FED_SIM_DIR}/clients_info.json") as f:
    clients_info = json.load(f)

FEATURE_COLS = json.load(open(f"{FED_SIM_DIR}/preproc_meta.json"))["feature_cols"]

import joblib
scaler = joblib.load(f"{FED_SIM_DIR}/scaler.pkl")

# ---------- Local training function ----------
def train_client(
    model, X, y,
    epochs=1,
    lr=1e-3,
    use_dp=False
):
    model.train()
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32)
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    privacy_engine = None
    if use_dp:
        privacy_engine = PrivacyEngine()
        model, opt, loader = privacy_engine.make_private(
            module=model,
            optimizer=opt,
            data_loader=loader,
            noise_multiplier=NOISE_MULTIPLIER,
            max_grad_norm=MAX_GRAD_NORM,
        )

    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logit = model(xb).squeeze()
            loss = F.binary_cross_entropy_with_logits(logit, yb)
            loss.backward()
            opt.step()

    eps = None
    if use_dp:
        eps = privacy_engine.get_epsilon(delta=DELTA)

    return model.state_dict(), eps


# ---------- FedAvg Training ----------
global_model = FedANN(len(FEATURE_COLS), EMB_SIZE).to(DEVICE)
start = time.time()
eps_history = []

print("\n🚀 Starting Federated Training...")

for rnd in range(N_ROUNDS):
    print(f"\n----- ROUND {rnd+1}/{N_ROUNDS} -----")
    local_states = []
    local_sizes = []
    round_eps = []

    for cid, info in clients_info.items():
        dfc = pd.read_csv(info["path"])
        Xc = scaler.transform(dfc[FEATURE_COLS].fillna(0))
        yc = dfc["pseudo_label"].values

        local_model = copy.deepcopy(global_model).to(DEVICE)
        state, eps = train_client(
            local_model, Xc, yc,
            epochs=LOCAL_EPOCHS,
            lr=LR,
            use_dp=USE_DP
        )

        local_states.append(state)
        local_sizes.append(len(dfc))
        if eps is not None:
            round_eps.append(eps)

    # FedAvg aggregation
    new_state = copy.deepcopy(global_model.state_dict())
    for k in new_state.keys():
        new_state[k] = sum(
            local_states[i][k] * local_sizes[i]
            for i in range(len(local_states))
        ) / sum(local_sizes)

    global_model.load_state_dict(new_state)

    if USE_DP and round_eps:
        eps_history.append(max(round_eps))
        print(f"🔐 ε (round max): {eps_history[-1]:.2f}")

    print("✔ Round aggregated.")

# ---------- Save global model ----------
torch.save(global_model.state_dict(), f"{FED_OUT_DIR}/global_fed_ann.pt")

dp_info = {
    "use_dp": USE_DP,
    "noise_multiplier": NOISE_MULTIPLIER if USE_DP else None,
    "max_grad_norm": MAX_GRAD_NORM if USE_DP else None,
    "delta": DELTA if USE_DP else None,
    "epsilon": max(eps_history) if eps_history else None
}

with open(f"{FED_OUT_DIR}/dp_info.json", "w") as f:
    json.dump(dp_info, f, indent=2)

print("\n🎉 FedAvg Complete!")
print("Saved global model →", f"{FED_OUT_DIR}/global_fed_ann.pt")
print("Total time (sec):", round(time.time() - start, 2))


Using device: cpu

🚀 Starting Federated Training...

----- ROUND 1/12 -----
✔ Round aggregated.

----- ROUND 2/12 -----
✔ Round aggregated.

----- ROUND 3/12 -----
✔ Round aggregated.

----- ROUND 4/12 -----
✔ Round aggregated.

----- ROUND 5/12 -----
✔ Round aggregated.

----- ROUND 6/12 -----
✔ Round aggregated.

----- ROUND 7/12 -----
✔ Round aggregated.

----- ROUND 8/12 -----
✔ Round aggregated.

----- ROUND 9/12 -----
✔ Round aggregated.

----- ROUND 10/12 -----
✔ Round aggregated.

----- ROUND 11/12 -----
✔ Round aggregated.

----- ROUND 12/12 -----
✔ Round aggregated.

🎉 FedAvg Complete!
Saved global model → /content/fed_output/global_fed_ann.pt
Total time (sec): 150.74


In [8]:
# ===== Cell 6: Extract ANN Embeddings + ANN Probabilities =====

import torch
import numpy as np
import pandas as pd
import joblib
import json
import os

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EMB_DIR = "/content/embeddings"
os.makedirs(EMB_DIR, exist_ok=True)

# ---------- Load global ANN ----------
model = FedANN(len(FEATURE_COLS), EMB_SIZE).to(DEVICE)
model.load_state_dict(torch.load(f"{FED_OUT_DIR}/global_fed_ann.pt", map_location=DEVICE))
model.eval()

print("✅ Loaded global FedANN")

# ---------- Load full pseudo-labeled dataset ----------
df = pd.read_csv(f"{PSEUDO_DIR}/full_with_pseudo_labels.csv")
X = scaler.transform(df[FEATURE_COLS].fillna(0))

# ---------- Forward pass ----------
all_emb = []
all_prob = []

BATCH = 8192

with torch.no_grad():
    for i in range(0, len(X), BATCH):
        xb = torch.tensor(X[i:i+BATCH], dtype=torch.float32).to(DEVICE)
        logits, emb = model(xb, return_emb=True)

        prob = torch.sigmoid(logits).cpu().numpy().ravel()
        emb = emb.cpu().numpy()

        all_prob.append(prob)
        all_emb.append(emb)

all_prob = np.concatenate(all_prob)
all_emb = np.vstack(all_emb)

print("Embeddings shape:", all_emb.shape)
print("ANN prob shape:", all_prob.shape)

# ---------- Save outputs ----------
np.save(f"{EMB_DIR}/emb_full.npy", all_emb)

df["ann_prob"] = all_prob
df.to_csv(f"{EMB_DIR}/full_with_ann_probs.csv", index=False)

print("✅ Saved:")
print(" - emb_full.npy")
print(" - full_with_ann_probs.csv")
print("\n📌 NEXT: Cell-7 → Train VST (LightGBM) on ANN embeddings")


✅ Loaded global FedANN
Embeddings shape: (555719, 64)
ANN prob shape: (555719,)
✅ Saved:
 - emb_full.npy
 - full_with_ann_probs.csv

📌 NEXT: Cell-7 → Train VST (LightGBM) on ANN embeddings


In [9]:
# ===== Cell 7: Train VST (LightGBM) on ANN Embeddings =====

!pip install --quiet lightgbm

import lightgbm as lgb
import numpy as np
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

EMB_DIR = "/content/embeddings"

# ---------- Load data ----------
X = np.load(f"{EMB_DIR}/emb_full.npy")
df = pd.read_csv(f"{EMB_DIR}/full_with_ann_probs.csv")

y = df["pseudo_label"].values

print("Embeddings:", X.shape)
print("Pseudo positives:", y.sum())

# ---------- Train/Val split ----------
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ---------- Handle class imbalance ----------
pos = y_tr.sum()
neg = len(y_tr) - pos
scale_pos_weight = neg / max(pos, 1)

print("scale_pos_weight:", round(scale_pos_weight, 2))

# ---------- LightGBM model ----------
params = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "n_estimators": 1000,
    "num_leaves": 31,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": scale_pos_weight,
    "verbosity": -1,
}

gbm = lgb.LGBMClassifier(**params)

gbm.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
)

# ---------- Validation metrics (pseudo labels) ----------
val_prob = gbm.predict_proba(X_val)[:, 1]

roc = roc_auc_score(y_val, val_prob)
pr  = average_precision_score(y_val, val_prob)

print(f"Val ROC AUC (pseudo): {roc:.4f}")
print(f"Val PR  AUC (pseudo): {pr:.4f}")

# ---------- Save model ----------
joblib.dump(gbm, f"{EMB_DIR}/vst_gbm_on_emb.pkl")
print("✅ Saved VST GBM -> vst_gbm_on_emb.pkl")

# ---------- Predict on full dataset ----------
df["gbm_prob"] = gbm.predict_proba(X)[:, 1]
df.to_csv(f"{EMB_DIR}/full_with_ann_probs_and_gbm.csv", index=False)

print("✅ Saved -> full_with_ann_probs_and_gbm.csv")
print("\n📌 NEXT: Cell-8 → ANN + VST Ensemble (Meta-Learner)")


Embeddings: (555719, 64)
Pseudo positives: 555
scale_pos_weight: 1000.3


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Val ROC AUC (pseudo): 0.9726
Val PR  AUC (pseudo): 0.0256
✅ Saved VST GBM -> vst_gbm_on_emb.pkl


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ Saved -> full_with_ann_probs_and_gbm.csv

📌 NEXT: Cell-8 → ANN + VST Ensemble (Meta-Learner)


In [10]:
# ===== Cell 8: ANN + VST Ensemble (Meta-Learner) =====

import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

ENS_DIR = "/content/ensemble_output"
os.makedirs(ENS_DIR, exist_ok=True)

# ---------- Load data ----------
df = pd.read_csv("/content/embeddings/full_with_ann_probs_and_gbm.csv")

X_meta = df[["ann_prob", "gbm_prob"]].values
y_meta = df["pseudo_label"].values

print("Meta features shape:", X_meta.shape)
print("Pseudo positives:", y_meta.sum())

# ---------- Optional: confidence weighting ----------
# Higher confidence pseudo-labels get more weight
w = (df["ann_prob"] + df["gbm_prob"]) / 2
w = np.clip(w, 0.05, 1.0)

# ---------- Train/Val split ----------
X_tr, X_val, y_tr, y_val, w_tr, w_val = train_test_split(
    X_meta, y_meta, w,
    test_size=0.2,
    random_state=42,
    stratify=y_meta
)

# ---------- Meta-learner ----------
meta = LogisticRegression(
    max_iter=3000,
    class_weight="balanced"
)

meta.fit(X_tr, y_tr, sample_weight=w_tr)

# ---------- Validation (pseudo labels) ----------
val_prob = meta.predict_proba(X_val)[:, 1]

roc = roc_auc_score(y_val, val_prob)
pr  = average_precision_score(y_val, val_prob)

print(f"Val ROC AUC (pseudo): {roc:.4f}")
print(f"Val PR  AUC (pseudo): {pr:.4f}")

# ---------- Save meta-learner ----------
joblib.dump(meta, f"{ENS_DIR}/meta_lr.pkl")
print("✅ Saved meta -> meta_lr.pkl")

# ---------- Predict on full dataset ----------
df["meta_prob"] = meta.predict_proba(X_meta)[:, 1]
df.to_csv(f"{ENS_DIR}/full_with_meta_probs.csv", index=False)

print("✅ Saved ensemble output -> full_with_meta_probs.csv")
print("\n📌 NEXT: Cell-9 → Final evaluation on TRUE labels")


Meta features shape: (555719, 2)
Pseudo positives: 555
Val ROC AUC (pseudo): 0.9927
Val PR  AUC (pseudo): 0.6258
✅ Saved meta -> meta_lr.pkl
✅ Saved ensemble output -> full_with_meta_probs.csv

📌 NEXT: Cell-9 → Final evaluation on TRUE labels


In [11]:
# ===== Cell 9: Final Evaluation on TRUE Labels =====

import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report
)

# ---------- Load ensemble output ----------
df = pd.read_csv("/content/ensemble_output/full_with_meta_probs.csv")

assert "is_fraud" in df.columns, "❌ TRUE labels (is_fraud) not found!"

y_true = df["is_fraud"].astype(int).values
scores = df["meta_prob"].values

print("Total samples:", len(df))
print("True frauds:", y_true.sum())

# ---------- ROC / PR ----------
roc = roc_auc_score(y_true, scores)
pr  = average_precision_score(y_true, scores)

print("\n📊 Overall metrics:")
print(f"ROC AUC : {roc:.4f}")
print(f"PR  AUC : {pr:.4f}")

# ---------- Threshold tuning (F1) ----------
best_f1, best_thr = 0, 0.5
for thr in np.linspace(0.05, 0.95, 181):
    preds = (scores >= thr).astype(int)
    tp = ((preds == 1) & (y_true == 1)).sum()
    fp = ((preds == 1) & (y_true == 0)).sum()
    fn = ((preds == 0) & (y_true == 1)).sum()
    if tp + fp == 0 or tp + fn == 0:
        continue
    prec = tp / (tp + fp)
    rec  = tp / (tp + fn)
    f1 = 2 * prec * rec / (prec + rec + 1e-9)
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

print(f"\nBest F1: {best_f1:.4f} at threshold={best_thr:.2f}")

# ---------- Classification report ----------
y_pred = (scores >= best_thr).astype(int)
print("\n📋 Classification Report (best threshold):")
print(classification_report(y_true, y_pred, digits=4))

# ---------- Precision@K ----------
def precision_at_k(y_true, scores, ks):
    idx = np.argsort(scores)[::-1]
    out = {}
    for k in ks:
        k = min(k, len(scores))
        out[k] = y_true[idx[:k]].sum() / k
    return out

ks = [10, 50, 100, 500, 1000, 2000]
p_at_k = precision_at_k(y_true, scores, ks)

print("\n🎯 Precision@K:")
for k, v in p_at_k.items():
    print(f"P@{k}: {v:.4f}")

# ---------- Save shortlist for bank review ----------
TOP_K_REVIEW = 500
review_df = df.iloc[np.argsort(scores)[-TOP_K_REVIEW:]].sort_values(
    "meta_prob", ascending=False
)
review_path = "/content/ensemble_output/top_500_for_bank_review.csv"
review_df.to_csv(review_path, index=False)

print("\n✅ Saved top candidates for bank review ->", review_path)


Total samples: 555719
True frauds: 2145

📊 Overall metrics:
ROC AUC : 0.8379
PR  AUC : 0.0723

Best F1: 0.1948 at threshold=0.84

📋 Classification Report (best threshold):
              precision    recall  f1-score   support

           0     0.9968    0.9979    0.9973    553574
           1     0.2349    0.1664    0.1948      2145

    accuracy                         0.9947    555719
   macro avg     0.6158    0.5822    0.5961    555719
weighted avg     0.9938    0.9947    0.9942    555719


🎯 Precision@K:
P@10: 0.0000
P@50: 0.0000
P@100: 0.0000
P@500: 0.1800
P@1000: 0.2190
P@2000: 0.1980

✅ Saved top candidates for bank review -> /content/ensemble_output/top_500_for_bank_review.csv
